# DataMate: chạy thử Qwen 4B trên Kaggle

1. Trong Kaggle, mở **Settings → Accelerator → GPU T4 x2** và bật **Internet**.
2. Dùng **Add Input** để thêm Kaggle Dataset chứa thư mục `train` và `validation` đã chia. Dataset nên để **Private**; không đưa dữ liệu cuộc thi lên GitHub.
3. Chạy lần lượt các ô bên dưới. Mặc định chạy 5 câu train; kết quả nằm ở `/kaggle/working/datamate_runs`.

Kaggle dùng GPU thứ nhất; GPU thứ hai chưa được sử dụng. Đây là notebook thử nghiệm, **chưa phải Docker nộp cuộc thi**. Code sinh ra có kiểm tra và timeout nhưng không phải sandbox hoàn chỉnh.


In [ ]:
from pathlib import Path
from datetime import datetime

SPLIT = 'train'
LIMIT = 5  # None: toàn bộ; đổi thành validation khi cần.
DATA_ROOT = None  # None tự tìm; hoặc điền đường dẫn chứa train/ và validation/.
MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
MAX_INPUT_TOKENS = 12288
MAX_NEW_TOKENS = 1024
MAX_REPAIRS = 3
EXEC_TIMEOUT = 60
SEED = 42
OUTPUT_DIR = Path('/kaggle/working/datamate_runs') / ('qwen_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
RESUME = False
print('Kết quả:', OUTPUT_DIR)


In [ ]:
import subprocess, sys, urllib.request
from urllib.parse import quote
import json

REPO = 'luwukien/zalo-ai-challenge'
CODE_REF = 'codex/datamate-baseline'
request = urllib.request.Request(f'https://api.github.com/repos/{REPO}/commits/{quote(CODE_REF, safe="")}', headers={'User-Agent': 'datamate-kaggle'})
with urllib.request.urlopen(request, timeout=60) as response:
    CODE_SHA = json.load(response)['sha']
local_code = Path('/kaggle/working/datamate_code')
local_code.mkdir(parents=True, exist_ok=True)
url = f'https://raw.githubusercontent.com/{REPO}/{CODE_SHA}/scripts/datamate_baseline.py'
with urllib.request.urlopen(url, timeout=60) as response:
    (local_code / 'datamate_baseline.py').write_bytes(response.read())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.5.0', 'accelerate>=1.0', 'bitsandbytes>=0.49', 'openpyxl>=3.1'])
print('Code commit:', CODE_SHA)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Settings → Accelerator → GPU T4 x2 rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))
if DATA_ROOT is None:
    candidates = sorted({p.parent.parent for p in Path('/kaggle/input').rglob('questions.jsonl') if p.parent.name == SPLIT and (p.parent / 'rulebook.md').is_file() and (p.parent / 'data').is_dir()})
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một thư mục dữ liệu cho {SPLIT}; tìm thấy {candidates}. Điền DATA_ROOT trong ô cấu hình.')
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
print('Dữ liệu:', DATA_ROOT / SPLIT)
if not (DATA_ROOT / SPLIT / 'questions.jsonl').is_file():
    raise FileNotFoundError('Không tìm thấy questions.jsonl. Kiểm tra Add Input và DATA_ROOT.')


In [ ]:
sys.path.insert(0, str(local_code))
import datamate_baseline as dm
config = dm.Config(data_root=str(DATA_ROOT), split=SPLIT, limit=LIMIT, seed=SEED,
                   max_repairs=MAX_REPAIRS, exec_timeout=EXEC_TIMEOUT,
                   max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
                   model_id=MODEL_ID, output_dir=str(OUTPUT_DIR), resume=RESUME)
MODEL = dm.QwenModel(config)
print('Đã nạp:', MODEL.info)


In [ ]:
def show_question(record):
    print('\nCâu:', record['question'])
    for attempt in record['attempts']:
        if 'code' in attempt:
            print('CODE:\n', attempt['code'])
        error = attempt.get('error') or attempt.get('execution', {}).get('error')
        if error:
            print('LỖI:', error)
    print('Thời gian:', record['timing'])

summary = dm.run_baseline(config, model=MODEL, on_result=show_question)
print('Trạng thái:', summary['status'])
print('Đã xong:', summary['completed'], '/', summary['requested'])
if summary.get('error'):
    print('LỖI:', summary['error'])
print('Thư mục kết quả:', OUTPUT_DIR)


In [ ]:
import pandas as pd
from IPython.display import display
report = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
references = {x['id']: x for x in report.get('reference_evaluation', {}).get('comparisons', [])}
rows = []
for trace in dm.read_jsonl(OUTPUT_DIR / 'trace.jsonl'):
    ref = references.get(trace['id'], {})
    rows.append({'câu hỏi': trace['question'], 'dự đoán': trace.get('prediction'),
                 'đáp án tham khảo': ref.get('reference'), 'khớp': ref.get('match'),
                 'trạng thái': trace['status'], 'giây': trace['timing'].get('total_seconds')})
display(pd.DataFrame(rows))
print('So sánh chỉ để tham khảo, không phải bộ chấm chính thức.')
print('Tải thư mục kết quả trong phần Output của Kaggle.')


## Chạy validation
Sửa `SPLIT = 'validation'`, `LIMIT = 20` (hoặc `None` để chạy hết), đổi `OUTPUT_DIR` sang thư mục mới và chạy lại các ô từ đầu. Không dùng public/private test để huấn luyện. Để tiếp tục lượt bị ngắt, đặt `OUTPUT_DIR` về thư mục cũ và `RESUME = True`; cấu hình và mã phải giống lần cũ.

Nếu báo CUDA out of memory, lưu nguyên thông báo lỗi và gửi lại. Giảm `MAX_INPUT_TOKENS` có thể khiến câu bị `ContextOverflow` vì rulebook dài. Trọng số model và dữ liệu đầu vào không được đưa vào Git.
